# Intent Classification (E2)


## 1. فحص البيانات وتجهيزها

- تحميل بيانات الشكاوى.
- مراجعة توزيع النيات الخمس وحالات التصعيد.
- التحقق من عدم وجود نصوص ناقصة أو فارغة أو مكررة.
- التأكد من أن قيم التصنيفات ثنائية (0 أو 1).
- اعتماد التصنيف متعدد النيات؛ إذ يمكن للنص حمل أكثر من نية.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GridSearchCV
import re
from sklearn.metrics import classification_report ,f1_score ,make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

In [ ]:
df = pd.read_csv("data/processed/callsense_merged_complete.csv")
df.head()
df.shape

In [ ]:
intent_cols = [
    "billing",
    "technical",
    "cancellation",
    "complaint",
    "information",
]

label_cols = intent_cols + ["escalation"]

print("عدد النصوص لكل تصنيف:")
print(df[label_cols].sum())

print("\nالقيم الناقصة:")
print(df[["model_text"] + label_cols].isna().sum())

In [ ]:
label_cols = intent_cols + ["escalation"]

print("هل جميع التصنيفات 0 أو 1؟")
print(df[label_cols].isin([0, 1]).all())

print("\nعدد النصوص الفارغة:")
print(df["model_text"].str.strip().eq("").sum())

print("\nعدد النصوص المكررة:")
print(df["model_text"].str.strip().duplicated().sum())

## 2. تقسيم البيانات

تقسيم الشكاوى إلى 80% للتدريب و20% للتحقق، مع الحفاظ على
توزيع النية الأساسية باستخدام stratify وتثبيت random_state لإعادة النتائج.

تُستخدم النية الأساسية لتوازن التقسيم فقط، بينما يعتمد التدريب على
أعمدة النيات الخمس والتصعيد. تُحفظ مكالمات الفريق المخصصة للاختبار
منفصلة عن التدريب وضبط النماذج.

In [ ]:
df["author"] = df["source_url"].str.extract(r"x\.com/([^/]+)/status/")[0]
df["author"] = df["author"].fillna(df["complaint_id"])

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, val_idx = next(
    sgkf.split(df, df["primary_intent"], df["author"])
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Shared authors:", len(set(train_df["author"]) & set(val_df["author"])))

In [ ]:
distribution = pd.DataFrame({
    "train": train_df[label_cols].sum(),
    "validation": val_df[label_cols].sum()
})

distribution

## 3. بناء النموذج المرجعي بالكلمات المفتاحية — Keyword Baseline

بناء قواعد بسيطة لتوقع النيات الخمس والتصعيد من الكلمات والعبارات
الموجودة في النص، مع السماح بتوقع أكثر من نية للنص الواحد.

تُطوّر القواعد باستخدام بيانات التدريب فقط، ثم تُقيّم على بيانات التحقق.
تُستخدم النتائج كنقطة مقارنة مع النماذج المتعلمة لاحقًا.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub("[أإآٱ]", "ا", text)
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي")
    text = text.replace("ـ", "")
    return text

In [ ]:
keywords = {
    "billing": ["فاتوره", "فواتير", "رسوم", "استرداد", "refund"],
    "technical": ["يفصل", "انقطاع", "بطيء", "ما يشتغل", "راوتر"],
    "cancellation": ["الغي", "الغاء", "ايقاف الاشتراك", "cancel"],
    "complaint": ["شكوي", "اشتكي", "سيئه", "غير راضي", "تعبت"],
    "information": ["كيف", "كم", "متي", "هل", "استفسار"],
    "escalation": ["مشرف", "مدير", "شكوي رسميه", "برفع شكوي"]
}

In [ ]:
def predict_keywords(text):
    text = clean_text(text)
    result = {}

    for label in keywords:
        result[label] = 0

        for word in keywords[label]:
            if clean_text(word) in text:
                result[label] = 1
                break

    return result

In [ ]:
predict_keywords("فاتورتي مرتفعة وأبغى ألغي الاشتراك")

In [ ]:
predictions = []

for text in val_df["model_text"]:
    predictions.append(predict_keywords(text))

pred_df = pd.DataFrame(predictions, index=val_df.index)

pred_df.head()

In [ ]:
# Display the first validation example
print(val_df.iloc[0]["model_text"])

# Display its actual labels
print(val_df.iloc[0][intent_cols])

# Display keyword baseline predictions
print(pred_df.iloc[0][intent_cols])


In [ ]:
intent_cols = [
    "billing",
    "technical",
    "cancellation",
    "complaint",
    "information"
]

print(classification_report(
    val_df[intent_cols],
    pred_df[intent_cols],
    target_names=intent_cols,
    zero_division=0
))

In [ ]:
print(classification_report(
    val_df["escalation"],
    pred_df["escalation"],
    labels=[0, 1],
    target_names=["No escalation", "Escalation"],
    zero_division=0
))

Intent Macro-F1: 0.55
Escalation Precision: 1.00
Escalation Recall: 0.33
Escalation F1: 0.50

In [ ]:
missed = val_df[
    (val_df["escalation"] == 1) &
    (pred_df["escalation"] == 0)
]

for _, row in missed.iterrows():
    print("ID:", row["complaint_id"])
    print("Text:", row["model_text"])
    print("-" * 50)


## 4. تصنيف النيات باستخدام TF-IDF وLogistic Regression

تحويل النصوص إلى تمثيل رقمي باستخدام TF-IDF، ثم تدريب مصنف
لكل نية باستخدام OneVsRestClassifier لدعم تعدد النيات.

يتعلم TF-IDF من بيانات التدريب فقط، وتُقيّم النتائج على بيانات التحقق.

In [ ]:
intent_pipeline = Pipeline([
    (
        "cleaning",
        FunctionTransformer(
            lambda texts: [clean_text(t) for t in texts],validate=False)),
    (
        "tfidf",
        TfidfVectorizer(ngram_range=(1, 2))
    ),
    (
        "classifier",OneVsRestClassifier(LogisticRegression(max_iter=1000,class_weight="balanced")
        ))])

intent_pipeline.fit(train_df["model_text"],train_df[intent_cols])
predictions = intent_pipeline.predict(val_df["model_text"])
print(classification_report(val_df[intent_cols],predictions,target_names=intent_cols,zero_division=0
))


In [ ]:
pred_df_lr = pd.DataFrame(
    predictions,
    columns=intent_cols,
    index=val_df.index
)

errors = (pred_df_lr != val_df[intent_cols]).any(axis=1)

print("عدد المنشورات التي فيها أخطاء:", errors.sum())

display(val_df.loc[errors, ["complaint_id", "model_text"]])


In [ ]:
for idx in val_df.index[errors]:
    print("ID:", val_df.loc[idx, "complaint_id"])

    actual = val_df.loc[idx, intent_cols]
    predicted = pred_df_lr.loc[idx, intent_cols]

    print("Actual:", actual[actual == 1].index.tolist())
    print("Predicted:", predicted[predicted == 1].index.tolist())
    print("-" * 50)


In [ ]:
probabilities = intent_pipeline.predict_proba(
    val_df["model_text"]
)

prob_df = pd.DataFrame(
    probabilities,
    columns=intent_cols,
    index=val_df.index
)

display(
    pd.concat(
        [
            val_df[["complaint_id"]],
            prob_df.round(3)
        ],
        axis=1
    )
)


In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

for threshold in thresholds:
    preds = (probabilities >= threshold).astype(int)

    score = f1_score(
        val_df[intent_cols],
        preds,
        average="macro",
        zero_division=0
    )

    print(f"Threshold: {threshold:.2f} | Macro F1: {score:.3f}")


In [ ]:
from sklearn.base import clone

char_pipeline = clone(intent_pipeline)

char_pipeline.set_params(
    tfidf=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5)
    )
)

char_pipeline.fit(
    train_df["model_text"],
    train_df[intent_cols]
)

char_predictions = char_pipeline.predict(
    val_df["model_text"]
)
print(
    "Character TF-IDF Macro F1:",
    f1_score(
        val_df[intent_cols],
        char_predictions,
        average="macro"
    )
)


In [ ]:
print(classification_report(
    val_df[intent_cols],
    char_predictions,
    target_names=intent_cols,
    zero_division=0
))

In [ ]:
char_pred_df = pd.DataFrame(
    char_predictions,
    columns=intent_cols,
    index=val_df.index
)

char_errors = (
    char_pred_df != val_df[intent_cols]
).any(axis=1)

print("عدد المنشورات التي فيها أخطاء:", char_errors.sum())

for idx in val_df.index[char_errors]:
    actual = val_df.loc[idx, intent_cols]
    predicted = char_pred_df.loc[idx, intent_cols]

    print("ID:", val_df.loc[idx, "complaint_id"])
    print("Actual:", actual[actual == 1].index.tolist())
    print("Predicted:", predicted[predicted == 1].index.tolist())
    print("-" * 40)


In [ ]:
char_pred_df = pd.DataFrame(
    char_predictions,
    columns=intent_cols,
    index=val_df.index
)

char_errors = (
    char_pred_df != val_df[intent_cols]
).any(axis=1)

print("عدد المنشورات التي فيها أخطاء:", char_errors.sum())

for idx in val_df.index[char_errors]:
    actual = val_df.loc[idx, intent_cols]
    predicted = char_pred_df.loc[idx, intent_cols]

    print("ID:", val_df.loc[idx, "complaint_id"])
    print("Actual:", actual[actual == 1].index.tolist())
    print("Predicted:", predicted[predicted == 1].index.tolist())
    print("-" * 40)


In [ ]:
# 1. Parameters to test
params = {
    "tfidf__ngram_range": [(2, 4), (3, 5), (3, 6)],
    "classifier__estimator__C": [0.1, 1, 10]
}

# 2. Group-aware cross-validation
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_splits = list(cv.split(
    train_df,
    train_df["primary_intent"],
    train_df["author"]
))

# 3. Grid Search
grid = GridSearchCV(
    estimator=char_pipeline,
    param_grid=params,
    scoring=make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    ),
    cv=cv_splits,
    n_jobs=-1
)

# 4. Find best parameters
grid.fit(
    train_df["model_text"],
    train_df[intent_cols]
)

# 5. Evaluate on validation
best_char_model = grid.best_estimator_

pred = best_char_model.predict(
    val_df["model_text"]
)

print("Best parameters:", grid.best_params_)

print(
    "Best CV Macro F1:",
    round(grid.best_score_, 4)
)

print(
    "Validation Macro F1:",
    round(
        f1_score(
            val_df[intent_cols],
            pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)

In [ ]:
probs = char_pipeline.predict_proba(val_df["model_text"])
y_true = val_df[intent_cols]

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
    preds = (probs >= threshold).astype(int)

    score = f1_score(
        y_true,
        preds,
        average="macro",
        zero_division=0
    )

    print(f"Threshold: {threshold:.2f} | Macro F1: {score:.4f}")

In [ ]:
escalation_pipeline = Pipeline([
    ("cleaning",FunctionTransformer(lambda texts: [clean_text(t) for t in texts],validate=False)),
    ("tfidf",TfidfVectorizer(analyzer="char_wb",ngram_range=(3, 5))),
    ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))
    ])

escalation_pipeline.fit(train_df["model_text"],train_df["escalation"])

esc_pred = escalation_pipeline.predict(val_df["model_text"])

print(classification_report(
    val_df["escalation"],esc_pred,labels=[0, 1],target_names=["No escalation", "Escalation"],zero_division=0
))


In [ ]:
esc_probs = escalation_pipeline.predict_proba(val_df["model_text"])[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = val_df["escalation"]

for threshold in [0.40, 0.42, 0.44, 0.45, 0.46, 0.48, 0.50]:
    y_pred = (esc_probs >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.2f} | "
        f"Recall: {recall:.2f} | "
        f"F1: {f1:.2f}"
    )


### ضبط عتبة التصعيد

جُرّبت عدة عتبات على بيانات التحقق. حققت العتبة 0.44 أعلى
F1 بين العتبات المجربة: 0.86، مع Recall يساوي 1.00
وPrecision يساوي 0.75.

اختيرت هذه العتبة مبدئيًا لأنها اكتشفت حالات التصعيد الثلاث
مع إنذار خاطئ واحد. يلزم تقييمها لاحقًا على مجموعة اختبار
منفصلة، خصوصًا لصغر مجموعة التحقق.

In [ ]:
best_threshold = 0.42
esc_pred = (esc_probs >= best_threshold).astype(int)

In [ ]:
escalation_threshold = 0.42

escalation_predictions = (
    esc_probs >= escalation_threshold
).astype(int)

print(classification_report(
    val_df["escalation"],
    escalation_predictions,
    labels=[0, 1],
    target_names=["No escalation", "Escalation"],
    zero_division=0
))

In [ ]:
old_train = train_df[
    ~train_df["complaint_id"].astype(str).str.startswith("NEW")
]

print("Old training samples:", len(old_train))
print("Validation samples:", len(val_df))

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score

LABELS = ["billing", "technical", "cancellation", "complaint", "information"]

# Model trained on old data only
old_model = clone(char_pipeline)
old_model.fit(old_train["model_text"], old_train[LABELS])

old_pred = old_model.predict(val_df["model_text"])

# Model trained on all available training data
new_model = clone(char_pipeline)
new_model.fit(train_df["model_text"], train_df[LABELS])

new_pred = new_model.predict(val_df["model_text"])

print("Old data Macro F1:",
      f1_score(val_df[LABELS], old_pred, average="macro", zero_division=0))

print("New data Macro F1:",
      f1_score(val_df[LABELS], new_pred, average="macro", zero_division=0))

In [ ]:
train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)